# SmartCampus: improved Colab GPU workflow

Qwen/Qwen3.5-4B only. Gemini 3.5 runs in the local VS Code project environment.
This clean reproduction notebook does not claim execution history. Actual runs,
attempts, source images, revisions and timing evidence are saved separately.

Select a Colab GPU. Upload `SmartCampus_Improved_Package.zip` and
`SmartCampus_Runtime_Final.zip` from the project's `run/comparison_v2/` folder.
The first is the original frozen source package; the second overlays documented
runtime/content fixes. Neither contains an API key. Public weights need no HF token.
To resume after Colab expires, also upload your saved
`SmartCampus_Improved_Qwen_Results.zip`. Its outputs and persistent ledger are
restored only into a fresh output directory; newer live work is never overwritten.
For only the separate source-review repair candidates, restore those results,
set `RUN_FULL_WORKFLOW=False`, and set `RUN_REVIEWED_REPAIRS=True` below.

Every guide chapter receives four original page images and their original text.
Source-based content checks and key-hidden quiz audits are model reviews, not
independent teacher approval. Flagged outputs remain drafts. Computation is
GPU-only; CPU/laptop inference is rejected. Free Colab availability is not guaranteed.


In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.23.0','accelerate==1.15.0','bitsandbytes==0.50.2','xgrammar==0.2.9','Pillow','jsonschema'],check=True)
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime; no CPU or laptop model inference.'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import zipfile, sys, json, hashlib
from google.colab import files
base=Path('/content')
needed=['SmartCampus_Improved_Package.zip','SmartCampus_Runtime_Final.zip']
if any(not (base/name).exists() for name in needed):
    print('Upload both packages through this chooser or the Files pane.')
    files.upload()
assert all((base/name).exists() for name in needed)
root=base/'smartcampus_comparison_v2';root.mkdir(exist_ok=True)
for name in needed:
    with zipfile.ZipFile(base/name) as z:
        for member in z.namelist():assert (root/member).resolve().is_relative_to(root.resolve())
        z.extractall(root)
saved=base/'SmartCampus_Improved_Qwen_Results.zip'
ledger=root/'outputs/qwen/request_ledger.json'
if saved.exists() and not ledger.exists():
    with zipfile.ZipFile(saved) as z:
        assert z.testzip() is None,'Damaged saved result archive'
        entries=[name for name in z.namelist() if name.startswith('outputs/qwen/') and not name.endswith('/')]
        # The ledger is restored last; repeated preparation never resets it.
        entries.sort(key=lambda name:name.endswith('/request_ledger.json'))
        for name in entries:
            target=(root/name).resolve();assert target.is_relative_to(root.resolve())
            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))
    print('Saved outputs and persistent request ledger restored.')
elif ledger.exists():
    print('Keeping existing live outputs and ledger.')
sys.path.insert(0,str(root))
print('Input images:',len(list((root/'images').glob('*.jpg'))))
print('Runtime SHA256:',hashlib.sha256((root/'improved.py').read_bytes()).hexdigest())


In [ ]:
import gc
from improved import Runner
from refine_improved_quiz import refine
RUN_FULL_WORKFLOW=True
if 'runner' in globals() and runner.root.resolve()==root.resolve() and runner.arm=='qwen':
    runner.__class__=Runner
    print('Reusing the loaded GPU model; weights are not loaded twice.')
else:
    if 'runner' in globals():
        del runner;gc.collect();torch.cuda.empty_cache()
    runner=Runner(root,'qwen')
print('Tokenizer/chat EOS:',runner.processor.tokenizer.eos_token_id,
      'default model EOS:',runner.model.generation_config.eos_token_id)
# Runner explicitly aligns stopping with the tokenizer/chat EOS.
# Existing completed records with matching inputs/settings are reused; ledgers persist.
if RUN_FULL_WORKFLOW:
    runner.run()
    refine(runner)
else:
    print('Full-workflow generation skipped; GPU runner is ready for targeted repairs.')


In [ ]:
# Optional supervised-development candidates, separate from matched v2 outputs.
# Source cards are assistant review feedback, not human instructor approval.
RUN_REVIEWED_REPAIRS=False
if RUN_REVIEWED_REPAIRS:
    from reviewed_source_repairs import repair_candidates
    repair_candidates(runner)
else:
    print('No supervised repair requests made.')


In [ ]:
archive=Path('/content/SmartCampus_Improved_Qwen_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in (root/'outputs/qwen').rglob('*'):
        if p.is_file():z.write(p,p.relative_to(root))
    for name in ['protocol.json','runtime_amendment.json','improved.py','common.py','lecture_quality.py','refine_improved_quiz.py','reviewed_source_repairs.py','source_review_cards.json']:
        z.write(root/name,name)
print('Archive SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
